A Jupyter notebook consists of two different types of cells:
- *Code cells* contain F$^\sharp$ code, while
- *Markdown cells* are used to write comments.

The current cell is a *markdown cell*, the next cell is a *code cell*.
This code cell loads the script `setup.fsx`, which defines how values are displayed in this notebook, and the style sheet `style.css`, which is only needed to format this notebook.

$$ \sqrt{a^2 + b^2} = c $$

In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# An Introduction to F$^\sharp$

## Evaluating expressions

F$^\sharp$ is a compiled language.  However, the .NET SDK contains *F$^\sharp$ Interactive*, an interactive shell that evaluates F$^\sharp$ code piece by piece.  The F$^\sharp$ kernel that runs this notebook is based on F$^\sharp$ Interactive.  Therefore, expressions can be evaluated directly.  In a *Jupyter*
notebook we just have to type <tt>Ctrl-Enter</tt> in the cell containing the expression.  Instead
of <tt>Ctrl-Enter</tt> we can also use <tt>Shift-Enter</tt>.

In [ ]:
1 + 2

## Basic Concepts: Type Annotations and Functions

Before introducing the individual data types and language constructs of F$^\sharp$, this section introduces two basic concepts that will recur throughout the chapter: type annotations and functions.  In addition, the keywords `let` and `mutable` for the definition of variables are introduced.

**Type Annotations**

F$^\sharp$ is a *statically typed* language: Every variable, every parameter, and the return value of every function has a type that is known at compile time.  A variable is defined with the keyword `let`, which binds a name to a value.  The type of the variable can be given explicitly by a so-called *type annotation*, which is noted with a colon (`:`) after the name of the variable:

In [ ]:
let name:   string = "Alice"  // Variable of type string
let age:    int    = 25       // Variable of type int
let active: bool   = true     // Variable of type bool

Usually, type annotations are optional, since F$^\sharp$ supports *type inference*: The compiler derives the type of a variable from the value that is assigned to it.  For example, when we define the variable `email` below, F$^\sharp$ infers that the type of `email` is `string`.

In [ ]:
let email = "karl.stroetmann@dhbw.de"

In F$^\sharp$, the symbol `=` has two meanings: In a `let` definition, it binds a name to a value, while everywhere else it is the *equality operator*.  Therefore, the next cell does not assign the number `42` to `email`, but compares `email` with `42`.  As a `string` cannot be compared with an `int`, the next cell yields an error.

In [ ]:
email = 42

**Immutable and Mutable Variables: `let` and `let mutable`**

Variables defined with `let` are *immutable*: Once a value has been bound to a name, this value cannot be changed.  The assignment operator of F$^\sharp$ is `<-`.  As `pi` is immutable, the next cell yields an error.

In [ ]:
let pi = 3.14159265  // immutable
pi <- 3.0            // Error: This value is not mutable.

If a variable is defined with `let mutable`, its value can be changed with the assignment operator `<-`.  In functional programming, mutable variables are rarely needed.

In [ ]:
let mutable counter = 0  // mutable
counter <- 1             // allowed
counter

**Functions**

A *function* is a named, reusable piece of code that computes a value from its arguments.  In F$^\sharp$, a function is defined with the keyword `let`, followed by the name of the function and its parameters.  The types of the parameters as well as the return type are provided with annotations.  The return type is written after the parameter list.  The body of the function follows the symbol `=`.  It is an *expression*, and there is no `return` keyword: the value of the body is the result of the function.  If the body consists of several lines, these lines are indented and the value of the last line is returned.  In general, F$^\sharp$ uses no curly braces to group code: the *indentation* of a line determines to which block it belongs.  If a function does not return a meaningful value, its return type is `unit`.  The type `unit` has only one value, which is written as `()`.

In the following code block, a function named `add` is defined, which takes two parameters `a` and `b` of type `int` and returns their sum, which is of type `int`, too.  The type `int` denotes a 32 bit signed integer.  Note that the parameters are *not* separated by commas: every parameter is enclosed in its own pair of parentheses.

In [ ]:
let add (a: int) (b: int) : int =
    a + b

A function is called by writing its arguments after its name.  The arguments are separated by spaces, too:

In [ ]:
add 3 5

As the arguments of `add` have to be integers, the next cell yields an error.

In [ ]:
add "ab" "cd"

A function with two parameters can also be called with just one argument.  The result is a function that still expects the second argument.  This is called *partial application*.  For example, `add 1` is a function that takes an integer and returns its successor.

In [ ]:
let inc = add 1
inc 41

**Anonymous Functions**

A function can also be defined without a name using the keyword `fun`.  To the left of the arrow (`->`) are the parameters, and to the right is the body of the function.  Such an *anonymous function* is also called a *lambda expression*.  It can be given a name by binding it to a variable, as shown below.  The function `sub` takes two integers `a` and `b` and returns their difference `a - b`.

In [ ]:
let sub = fun (a: int) (b: int) -> a - b

In [ ]:
sub 3 5

The type of a function is written with arrows.  For example, the type of both `add` and `sub` is `int -> int -> int`: they take an `int`, then another `int`, and return an `int`.

Anonymous functions are often used for short functions that are passed as arguments to other functions, in particular to the functions `List.filter`, `List.map`, and `List.fold`, which are introduced in the section on functional programming.

## Primitive Data Types and Arithmetic

In TypeScript and JavaScript, all numbers are floating-point numbers.  Unlike these languages, F$^\sharp$ distinguishes between integers and floating-point numbers.  The most important primitive data types are:

| **Type** | **Values** | **Literals** |
| :--- | :--- | :--- |
| `int` | integers with 32 bits | `42` |
| `int64` | integers with 64 bits | `42L` |
| `bigint` | integers of arbitrary size | `42I` |
| `float` | floating-point numbers with 64 bits according to the [IEEE-754 standard](https://en.wikipedia.org/wiki/IEEE_754) | `42.0`, `4.2e1` |
| `char` | single characters, more precisely 16 bit Unicode code units | `'a'` |
| `string` | sequences of characters | `"abc"` |
| `bool` | the truth values `true` and `false` | `true` |

Note that a floating-point literal has to contain a decimal point or an exponent: `42` is an `int`, while `42.0` is a `float`.  F$^\sharp$ uses the types of the .NET platform.  The method `GetType()` returns the type of a value at runtime.  At runtime, the type `int` is called `System.Int32` and the type `float` is called `System.Double`.

In [ ]:
(42).GetType()

In [ ]:
(42.123).GetType()

The range of every integer type is limited.  For example, the largest value of type `int` is $2^{31} - 1$.  If this limit is exceeded, the result *wraps around* silently: This is called an *overflow*.

In [ ]:
System.Int32.MaxValue

In [ ]:
System.Int32.MaxValue + 1

In [ ]:
System.Int64.MaxValue

The precision of the type `float` is limited, too.  Integers are only represented exactly as long as they are not bigger than $2^{53}$.  Beyond this limit, rounding errors occur, as shown below.

In [ ]:
9007199254740992.0 = 9007199254740993.0

To bypass these limitations, the type `bigint` can be used.  It allows the representation of integers of arbitrary size, limited only by the available memory.  Literals of type `bigint` are denoted by the suffix `I`.  Additionally, the function `bigint` converts a given number into a `bigint`.  The usual arithmetic operators can be used for values of type `bigint`.  The operator `**` computes a power.

In [ ]:
bigint 1

In [ ]:
let result: bigint = 7I ** 1235  // Calculation of extremely large powers

In [ ]:
result

When two integers are divided, the result is always an integer and hence is *truncated* sometimes.
Note that the result is not **rounded** to the nearest integer, but rather it is *rounded towards zero*.  This is true both for `int` and for `bigint`.  If both operands are floating-point numbers, the result is a floating-point number.

In [ ]:
19 / 4

In [ ]:
19.0 / 4.0

In [ ]:
-19 / 4

In [ ]:
-19I / 4I

In arithmetic expressions, F$^\sharp$ does not convert numbers of one type into another type automatically.  Hence, we can't mix the types `bigint` and `int` in arithmetic expressions, and the next cell yields an error.

In [ ]:
2I * 3

The same holds for the types `int` and `float`.  The next cell yields an error, too.

In [ ]:
1 + 2.0

In cases like these, we have to convert the `int` into a `bigint` using the function `bigint` or into a `float` using the function `float`.  F$^\sharp$ converts an `int` into a `float` automatically only if the expected type is known, for example if an integer is passed to a function whose parameter has the type annotation `float`.

In [ ]:
2I * bigint 3

In [ ]:
float 1 + 2.0

## Output and String Literals

For outputting values to the console, F$^\sharp$ provides the function `printfn`.  Its first argument is a *format string*, which can contain *format specifiers* like `%d` for integers, `%f` for floating-point numbers, `%c` for characters, `%s` for strings, and `%A` for arbitrary values.  These format specifiers are replaced by the remaining arguments of `printfn`, and the compiler checks that the types of these arguments match the format specifiers.  Finally, `printfn` appends a newline.

The data type for textual data is `string`.  In F$^\sharp$, string literals are enclosed in double quotes (`"`).  Single quotes (`'`) are used for literals of type `char`, which represent a single character.

In the interactive environment of a Jupyter Notebook, the result of the last expression of a cell is automatically output without an explicit call to `printfn` being necessary.

In [ ]:
// Explicit output via the console
printfn "1. Hello, World!"

// A format string with the format specifiers %d and %c
printfn "%d. Hello, World%c" 2 '!'

// Implicit output of the last expression of a code cell
"Hello, World!"

**Interpolated Strings**

In addition to classic string literals, F$^\sharp$ supports so-called *interpolated strings*.  These are string literals that are preceded by the character `$`.  They offer *string interpolation*, which allows expressions to be embedded directly into a string.  The syntax `{...}` is used here, where the expression in the curly braces is evaluated and the result is converted into a *string*.  This significantly increases the readability of the code, as complex concatenation operations are avoided.  Strings can also be concatenated with the operator `+`, and the function `string` converts an arbitrary value into a string.

Note that the following cell defines the variables `name` and `age` a second time.  These new definitions *shadow* the definitions given above: The old variables are not changed, but they can't be accessed by their names anymore.

In [ ]:
let name = "Alice"
let age  = 25

// Direct embedding of variables
printfn $"User {name} is {age} years old."

// Evaluation of expressions within the string
printfn $"Next year, she will be {age + 1}."

// Concatenation of strings
printfn "%s" ("User " + name + " is " + string age + " years old.")

**Multi-line Strings**

String literals can span multiple lines.  If a string contains double quotes, it is best written as a *triple-quoted string*, which is enclosed in `"""`.  Inside a triple-quoted string, the character `"` needs no escaping.

In [ ]:
let poem = """I am a teacher by profession,
mostly really by obsession,
but even though I boldly try,
I just cannot teach "pigs" to fly."""
printfn "%s" poem

## Arithmetic Operations and Variable Assignment

F$^\sharp$ provides the usual arithmetic operators.  These include addition (`+`), subtraction (`-`), multiplication (`*`), division (`/`), as well as the exponentiation operator (`**`) and the modulo operator (`%`), which determines the remainder of a division.  The operator `**` works for floating-point numbers and for values of type `bigint`, but not for values of type `int`.  For integers, there is the function `pown`, which computes the power $b^n$ of a number $b$ for an exponent $n$ of type `int`.

In [ ]:
1 + 2

In [ ]:
5 - 3

In [ ]:
4 * 3

In [ ]:
7 / 2      // integer division

In [ ]:
7.0 / 2.0  // floating point division

In [ ]:
7 % 3      // Modulo

In [ ]:
2.0 ** 10.0  // Exponentiation

In [ ]:
pown 2 10    // Exponentiation of integers

In [ ]:
2.0 ** 0.5

A significant difference compared to TypeScript and Python concerns division.  In F$^\sharp$, the division operator `/` performs an *integer division* if both operands are integers.  To perform a floating-point division of two integers, both have to be converted into floating-point numbers using the function `float`.

Integer division rounds towards zero.  If the result is negative, this differs from rounding down.  The function `floor`, which is discussed in the section on numerical functions, rounds a floating-point number down.  Similarly, the remainder computed by `%` has the sign of the dividend.

In [ ]:
float 7 / float 3     // Floating point division

In [ ]:
-7 / 2                // Integer division, rounds towards zero

In [ ]:
floor (-7.0 / 2.0)    // Rounding down

In [ ]:
-7 % 3                // Remainder

## Lists and Arrays

**Lists**

*Lists* are the most important data structure of functional programming.  A list is a sequence of elements of the same type.  In F$^\sharp$, a list is written in square brackets "`[`" and "`]`", and its elements are separated by *semicolons*.  The empty list, i.e., a list without elements, is written as "`[]`".  The type of a list of integers is written as `int list`.

In [ ]:
[1; 2; 3]

Be careful not to separate the elements of a list by commas: In F$^\sharp$, commas create a *tuple* (see below).  Hence, the following list does not contain three integers, but only a single element, which is the tuple `(1, 2, 3)`.

In [ ]:
[1, 2, 3]

**Accessing List Elements: The Index Operator**

The *index operator* `[·]` is a postfix operator used to access the element at a specific position (index).  F$^\sharp$, like most modern programming languages, uses zero-based indexing, meaning the first element of a list has the index `0`.  There must not be a space between the list and the index operator, since `xs [0]` would be read as the application of `xs` to the list `[0]`.

In [ ]:
let xs = [1; 2; 3]
xs[0]  // Returns the first element

Lists are *immutable*: Once a list has been created, its elements cannot be changed.  Hence, the next cell yields an error.

In [ ]:
xs[0] <- 7

Accessing an index that does not exist yields an exception.  Unlike TypeScript, F$^\sharp$ has no value `undefined`.  In particular, every variable has to be given a value when it is defined.

In [ ]:
xs[10]

If we are not sure whether an index exists, we can use the function `List.tryItem`.  It returns a value of the type `int option`: The value `Some x` means that the element at the given index is `x`, while the value `None` means that there is no such element.  The type `option` is the F$^\sharp$ replacement for the values `undefined` and `null`.

In [ ]:
List.tryItem 1 xs

In [ ]:
List.tryItem 10 xs

**Concatenating Lists**

Lists can be concatenated using the operator `@`.  This operator creates a new list containing all elements of the first list followed by all elements of the second list.  As lists are immutable, the list `xs` remains unchanged.

In [ ]:
[1; 2; 3] @ [4; 5; 6]

In [ ]:
xs @ [4; 5; 6]

In [ ]:
xs

The operator `::` (pronounced *cons*) creates a new list by putting a single element in front of a list.

In [ ]:
0 :: xs

Alternatively, lists can be combined in a *list expression*: Inside the square brackets, `yield!` inserts all elements of a list.  This is the F$^\sharp$ counterpart of the *spread operator* "`...`" of TypeScript.

In [ ]:
[yield! [1; 2; 3]; 0; yield! [4; 5; 6]]

The function `List.length` returns the number of elements of a list.  Alternatively, we can use the property `Length`.

In [ ]:
List.length [4; 5; 4]

In [ ]:
xs.Length

**Arrays**

Besides lists, F$^\sharp$ has *arrays*.  An array is written in the brackets "`[|`" and "`|]`".  Unlike lists, arrays are *mutable*: The index operator can also be used on the left side of an assignment to modify an existing element.  However, the length of an array is fixed when it is created.

In [ ]:
let arr = [|1; 2; 3|]
arr[0] <- 7
arr

As the length of an array is fixed, we cannot add an element at the end of an array.  The next cell yields an `IndexOutOfRangeException`.

In [ ]:
arr[3] <- 8

As all elements of a list or an array must have the same type, the next cell yields an error.

In [ ]:
[1; 2; "Hugo"]

## Functional Programming with Lists

The module `List` provides a number of functions that enable *functional programming*.  The most important ones are:
* `List.filter`,
* `List.map`,
* `List.fold`,
* `List.exists`, and `List.forall`.

These functions take a function as their first argument and a list as their last argument.  They are often combined with the *pipe operator* `|>`: The expression `x |> f` is the same as `f x`.  Hence, the list can be written in front of the function that is applied to it.

**The Function `List.filter`**

The function `List.filter` creates a new list that contains only those elements of the original list for which a passed condition is met.  The original list is not modified.  The condition is passed as a function that returns either `true` (element is kept) or `false` (element is discarded) for each element.  Usually, an *anonymous function* is used for such short conditions.

In the following example, only numbers greater than 3 are filtered out of a list of numbers:

In [ ]:
let numbers = [1; 2; 3; 4; 5]
let result  = numbers |> List.filter (fun n -> n > 3)
result

The function `fun n -> n > 3` is called internally for each element of the list.  For `n = 1`, `2`, and `3`, it returns `false`, which is why these elements are discarded.  For `n = 4` and `5`, it returns `true`, so they remain in the result list.  The original list `numbers` remains unchanged.

| **Element (n)** | **Condition (n > 3)** | **Result** |
| :--- | :--- | :--- |
| 1 | false | Drop |
| 2 | false | Drop |
| 3 | false | Drop |
| 4 | true | Keep |
| 5 | true | Keep |

**The Function `List.map`**

The function `List.map` transforms each element of a list using a passed function and returns a new list containing the transformed values.  The length of the resulting list is always identical to that of the original list.

In the following example, each element of a list of numbers is doubled:

In [ ]:
let numbers = [1; 2; 3; 4]
let result  = numbers |> List.map (fun n -> n * 2)
result

The function `fun n -> n * 2` is called once for each element and returns its doubled value.  The result is a new list where each position contains the transformed value of the corresponding position in the original.  The original list `numbers` remains unchanged.

**The Function `List.fold`**

The function `List.fold` condenses a list into a single value by applying a function to each element along with a running intermediate result (the so-called *accumulator*).  In TypeScript, this function is called `reduce`.

The call `List.fold (fun acc n -> acc + n) 0 numbers` consists of the following parts:
1. The **combining function**: `fun acc n -> acc + n` defines the calculation rule.
2. The **initial value**: The `0` is the initial value of the accumulator for the calculation.
3. The **parameters** of the combining function:
    * `acc` (short for *Accumulator*): Stores the intermediate result of all previous additions.
    * `n`: Is the current element from the list that is currently being processed.

Note that the initial value is the *second* argument of `List.fold`, while the list is the last argument.  Using the pipe operator, the call can also be written as `numbers |> List.fold (fun acc n -> acc + n) 0`.

In the following example, the sum of all elements of a list is calculated:

In [ ]:
let numbers = [1; 2; 3; 4]
let sum     = numbers |> List.fold (fun acc n -> acc + n) 0
sum

The accumulator `acc` starts with the value `0`.  In each step, the current element `n` is added to the previous accumulator value, and the result is passed on as the new accumulator value:

| **Step** | **Accumulator (acc)** | **Element (n)** | **New acc** |
| :--- | :--- | :--- | :--- |
| Start | 0 | --- | 0 |
| 1 | 0 | 1 | 1 |
| 2 | 1 | 2 | 3 |
| 3 | 3 | 3 | 6 |
| 4 | 6 | 4 | 10 |

**The Function `List.exists`**

The function `List.exists` checks whether there is at least one element that satisfies a given condition.  For example,
below we check whether `numbers` contains a number divisible by `13`.

In [ ]:
let numbers = [15; 30; 60; 39]
numbers |> List.exists (fun n -> n % 13 = 0)

**The Function `List.forall`**

The function `List.forall` checks whether all elements of a list satisfy a given condition.  For example,
below we check whether all numbers contained in `numbers` are even.

In [ ]:
let numbers = [2; 4; 6; 8; 10]
numbers |> List.forall (fun n -> n % 2 = 0)

**Creating Lists with Ranges, `List.init`, and List Comprehensions**

Besides listing the elements, F$^\sharp$ offers several ways to create lists.  This is particularly useful when a list of a specific length needs to be populated with calculated values, for example, to generate a sequence of numbers.

The simplest way is a *range*: The expression `[a .. b]` creates the list of all integers from `a` up to and including `b`.

In [ ]:
[1 .. 10]

The function `List.init` takes two arguments: first, the desired length of the list, and second, a function that is called for each index and determines the value at that position.  In the following example, a list with the numbers 1 to 10 is created:

In [ ]:
List.init 10 (fun i -> i + 1)

The index `i` runs from `0` to `9` and is transformed by `i + 1` into the desired values `1` to `10`.

Finally, lists can be created by *list comprehensions*.  For example, the following expression computes the list of the first 10 square numbers:

In [ ]:
[for i in 1 .. 10 -> i * i]

A list comprehension can also contain conditions.  The following list contains only the even square numbers.

In [ ]:
[for i in 1 .. 10 do if i % 2 = 0 then i * i]

**Pattern Matching: Head and Tail of a List**

A powerful feature of F$^\sharp$ is *pattern matching*.  It allows unpacking values from a data structure directly into separate variables.  Every non-empty list consists of its first element, the *head*, and the list of the remaining elements, the *tail*.  The pattern `head :: tail` matches every non-empty list: it binds the variable `head` to the first element and the variable `tail` to the list of the remaining elements.  Separating the *head* and the *tail* of a list is a common pattern in functional programming.

The expression `match` compares a value with a sequence of patterns.  Each pattern starts with a vertical bar `|` and is followed by an arrow `->` and the code that is executed if the pattern matches.  The first pattern that matches is used.  Below, the pattern `[]` matches the empty list.

In [ ]:
match [10; 20; 30; 40; 50] with
| head :: tail -> printfn "head = %d, tail = %A" head tail
| []           -> printfn "The list is empty."

The functions `List.head` and `List.tail` compute the head and the tail of a list directly.  They throw an exception if the list is empty.

In [ ]:
List.head [10; 20; 30; 40; 50]

In [ ]:
List.tail [10; 20; 30; 40; 50]

## Tuples

A *tuple* in F$^\sharp$ groups a fixed number of values, which can have different types.  The elements of a tuple are separated by commas and are usually enclosed in parentheses.  Unlike lists, which are homogeneously typed (e.g., `string list`), tuples allow the storage of a heterogeneous collection of values in a strict order.  This concept is particularly suitable for grouping related data without having to define a type of its own, for example for coordinate pairs `(x, y)` or for functions returning several values.  The type of a tuple is written with the symbol `*` between the types of its elements.

The type `bool` denotes a truth value and can only take the two values `true` or `false`.  A more detailed treatment of boolean values and operators is provided in the section *Boolean Values and Operators*.

In the following example, the name `UserProfile` is defined as an abbreviation of the tuple type `int * string * bool`.  The values of this type are tuples with exactly three elements in the order `int`, `string`, and `bool`:

In [ ]:
type UserProfile = int * string * bool                // Type abbreviation
let userProfile: UserProfile = (101, "Alice", true)  // Order and types must match exactly

There is no index operator for tuples.  Instead, the elements of a tuple are extracted by *pattern matching*: The left side of a `let` definition can be a pattern.  The *wildcard* `_` matches every value without binding it to a variable.

In [ ]:
let (_, userName, _) = userProfile
$"Name: {userName}"

For *pairs*, i.e. tuples with two elements, the functions `fst` and `snd` return the first and the second element.

In [ ]:
fst (1, "one")

In [ ]:
snd (1, "one")

## Records

In TypeScript, *objects* group values as *properties*.  In F$^\sharp$, the counterpart of these objects are *records*.  A record type groups a fixed number of values, the so-called *fields*, each of which has a name and a type.  Unlike the elements of a tuple, the fields of a record are accessed by their names.  By convention, the names of the fields start with a capital letter.

A record type is defined with the keyword `type`, and its fields are enclosed in curly braces `{}`.  The following example shows a record type `Student` with basic fields:

In [ ]:
type Student =
    { FirstName           : string
      MatriculationNumber : int
      IsEnrolled          : bool }

A record is created by listing the values of all fields in curly braces.  The compiler infers the type `Student` from the names of the fields.

In [ ]:
let student = { FirstName           = "Marvin"
                MatriculationNumber = 123456
                IsEnrolled          = true }

In [ ]:
student

**Accessing Fields**

The fields of a record are accessed with the dot notation `record.Field`.  The names of the fields are fixed when the record type is defined.  Unlike TypeScript, F$^\sharp$ has no bracket notation `obj[expr]` that computes the name of a field at runtime.

In [ ]:
student.FirstName

The fields of a record are *immutable*: They cannot be changed after the record has been created.  Hence, the next cell yields an error.

In [ ]:
student.IsEnrolled <- false

Instead, a *copy-and-update expression* creates a new record that differs from a given record only in the fields listed after the keyword `with`.  The original record remains unchanged.

In [ ]:
let formerStudent = { student with IsEnrolled = false }
formerStudent

If a field has to be changeable, it has to be declared with the keyword `mutable`.

In [ ]:
type Course =
    { Title                : string
      mutable Participants : int }

In [ ]:
let logic = { Title = "Logic"; Participants = 60 }
logic.Participants <- 61
logic.Participants

**Maps**

If the keys are only known at runtime, we use a *map* instead of a record.  A map stores *key-value pairs*.  It is created by applying the function `Map` to a list of pairs.  The value associated with a key `k` in a map `m` is accessed with the index operator `m[k]`.  Like lists, maps are immutable: The function call `Map.add k v m` returns a new map in which the key `k` is associated with the value `v`.  In this lecture, maps are used, e.g., to represent variable assignments.

In [ ]:
let zipCodes = Map [("Konstanz", 78462); ("Mannheim", 68163)]
let key = "Mannheim"
zipCodes[key]

In [ ]:
Map.add "Stuttgart" 70173 zipCodes

**Nested Records**

Records can be nested to map complex data structures.  The `University` example demonstrates a hierarchy where the field `Location` is itself a record and `Departments` is a list of strings.

In [ ]:
type Location =
    { Street  : string
      City    : string
      ZipCode : int }

type University =
    { Name        : string
      Location    : Location
      Departments : string list
      IsPublic    : bool }

In [ ]:
let university =
    { Name        = "HTWG Konstanz"
      Location    = { Street = "Alfred-Wachtel-Str. 8"; City = "Konstanz"; ZipCode = 78462 }
      Departments = ["Computer Science"; "Architecture"; "Mechanical Engineering"]
      IsPublic    = true }

Access to deeper levels is achieved by chaining the dot operator or combining it with the index operator:

In [ ]:
// Accessing nested record
university.Location.City

In [ ]:
// Accessing list within a record
university.Departments[1]

## Discriminated Unions

Often, a value can have one of several different forms.  For example, a geometric shape might be either a circle, which is given by its radius, or a rectangle, which is given by its width and its height.  In F$^\sharp$, such data are modeled by *discriminated unions*.  The definition of a discriminated union lists its *cases*.  Each case starts with a vertical bar `|` and has a name starting with a capital letter.  A case can carry data, whose type is written after the keyword `of`.

In [ ]:
type Shape =
    | Circle    of float
    | Rectangle of float * float

A value of a discriminated union is created by applying the name of a case to its data:

In [ ]:
let shapes = [Circle 1.0; Rectangle (2.0, 3.0)]

Functions on discriminated unions are defined by pattern matching.  The function `area` takes a shape and returns its area.  It has one branch for every case of `Shape`.  The patterns of these branches bind the data of the cases to variables.  The compiler warns if a case is forgotten.  In later chapters, formulas are represented as discriminated unions and most functions on formulas are defined in this way.

In [ ]:
let area (shape: Shape) : float =
    match shape with
    | Circle r         -> System.Math.PI * r * r
    | Rectangle (w, h) -> w * h

In [ ]:
shapes |> List.map area

## Boolean Values and Operators

The representation of truth values in F$^\sharp$ is done via the data type `bool`, which can only take the values `true` and `false`.  These values are central for controlling the program flow and for logical decisions.

**Logical Operators and Lazy Evaluation**

F$^\sharp$ supports the classic logical operations conjunction (AND), disjunction (OR), and negation (NOT):
* `a && b` is the conjunction of `a` and `b`,
* `a || b` is the disjunction of `a` and `b`, and
* `not a` is the negation of `a`.  Note that `not` is a function, not an operator.

An essential concept in the evaluation of `&&` and `||` is *lazy evaluation* (often referred to as *short-circuit evaluation* in this context).  This means that the second operand is only evaluated if it is absolutely necessary for the final result:
* For `A && B`, `B` is no longer evaluated if `A` is already `false` (the result is definitely `false` anyway).
* For `A || B`, `B` is no longer evaluated if `A` is already `true` (the result is definitely `true` anyway).

F$^\sharp$ has no operator for the *exclusive or*.  However, for truth values the operator `<>` can be used instead, since `a <> b` is `true` if and only if exactly one of the operands `a` and `b` is `true`.

In [ ]:
true && false

In [ ]:
true || false

In [ ]:
not true

In [ ]:
true <> true

**Comparison Operators and Equality**

Comparisons are frequently used to generate truth values.  In F$^\sharp$, the operator `=` checks whether two values are equal and the operator `<>` checks whether they are different.  There are no operators `==`, `===`, `!=`, or `!==`.  F$^\sharp$ does not convert strings into numbers or numbers into strings.  Hence, the expression `5 = "5"` is not *well-typed* and is rejected by the compiler.

In [ ]:
5 = "5"

The operator `=` compares values *structurally*: Two lists are equal if they contain the same elements in the same order, and two records are equal if all their fields are equal.  This holds even if the two values are stored as different objects in memory.

In [ ]:
[1; 2; 3] = [1; 2] @ [3]

In [ ]:
student = { formerStudent with IsEnrolled = true }

The classic size comparisons (`<`, `>`, `<=`, `>=`) are available in F$^\sharp$.

An important difference compared to Python is the chaining of comparisons.  A mathematical expression like $3 > 2 > 1$ is not accepted in F$^\sharp$, because the result of `3 > 2` is a `bool` and we can't compare a `bool` with an `int`.  Therefore, mathematical range checks must be explicitly linked with the *logical and operator*: `3 > 2 && 2 > 1`.

In [ ]:
3 > 2 > 1

In [ ]:
3 > 2 && 2 > 1

## Control Structures

Control structures serve to control the flow of a program.  In F$^\sharp$, most control structures are *expressions*, i.e. they have a value.

**Branches**

The basic conditional execution is done using `if`, `elif`, and `else`.  Unlike TypeScript, the conditions are not placed in parentheses, but they are followed by the keyword `then`.  The conditions must have the type `bool`.  There are no curly braces: The branches are determined by the indentation of the code.

In [ ]:
let x = 17
if x < 0 then
    printfn "The number is negative!"
elif x = 0 then
    printfn "The number is zero."
else
    printfn "The number is positive."

As `if` is an expression, it can be used to compute a value.  In this case, all branches must have values of the same type and the `else` branch is mandatory.

In [ ]:
let sign = if x < 0 then -1 elif x = 0 then 0 else 1
sign

**The `match` Expression**

Instead of a `switch` statement, F$^\sharp$ has the `match` expression, which we have already used to split a list into its head and its tail.  It compares a value with several patterns.  Unlike the `switch` statement of TypeScript, `match` does not "fall through": Only the branch of the first matching pattern is executed, so no `break` is needed.  A pattern can be a constant, and it can be combined with a condition that is introduced by the keyword `when`.  The pattern `n when n < 0` matches every number `n` that is negative.  The wildcard `_` matches every value.

In [ ]:
let y = -5
let description =
    match y with
    | 0            -> "The number is zero."
    | n when n < 0 -> "The number is negative!"
    | _            -> "The number is positive."
description

**Loop Constructs**

F$^\sharp$ provides various loop constructs for iteration.  Loops have no meaningful value, i.e. their value is `()`.  Therefore, they are used for their side effects, e.g. to print something or to change a mutable variable or an array.
* The **counting `for` loop** `for i = a to b do` lets the variable `i` run from `a` up to and including `b`.

In [ ]:
for i = 0 to 2 do
    printfn "%d" i

* The **`for ... in` loop** is ideally suited for iterating directly over the elements of a list, an array, or a range like `0 .. 2`.

In [ ]:
let numbers = [1; 2; 3]
for element in numbers do
    printfn "%d" element

* The **`while` loop** executes a block of code as long as a defined running condition is met.  Here, the loop variable has to be declared as `mutable`.

In [ ]:
let arr = [|1; 2; 3|]
let mutable i = 0
while i < arr.Length do
    printfn "%d" arr[i]
    i <- i + 1

F$^\sharp$ has no `break` and no `continue`.  Instead of using `continue` to skip the rest of the current iteration, the body of the loop is put into an `if`.

In [ ]:
let arr = [|1; 2; 3|]
for i in 0 .. arr.Length - 1 do
    if i <> 1 then
        printfn $"arr[{i}] = {arr[i]}"

**Application Example: [Sieve of Eratosthenes](https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes)**

A classic algorithm for determining all prime numbers up to a given limit $n\in\mathbb{N}$ is the Sieve of Eratosthenes.  The implementation in F$^\sharp$ demonstrates the interaction of arrays, the `for` loop as well as the `while` loop, and conditional statements:

1. Initialization of an array `primes`, where `primes[i] = i`.
2. Iteration over all numbers $i$ and $j$ to eliminate multiples (i.e., set them to 0).
3. Filtering the remaining numbers $\neq 0$.

An optimized version also uses $\lceil \sqrt{n} \rceil$ as the upper limit for the outer loop and starts the inner loop at $j=i$ to avoid redundant checks.

The function `primes` takes one argument:
- `n` is a natural number.

It returns the list of all prime numbers that are less than or equal to `n`.  The expression `Array.init (n + 1) id` creates the array `[|0; 1; ...; n|]`, since the function `id` returns its argument unchanged.  As the functions `sqrt` and `ceil` work with floating-point numbers, `n` is converted into a `float` and the result is converted back into an `int` using the function `int`.  Finally, the function `List.ofArray` converts the filtered array into a list.

In [ ]:
let primes (n: int) : int list =
    // Initialize array with values 0 to n
    let primes = Array.init (n + 1) id
    let limit  = int (ceil (sqrt (float n)))
    for i in 2 .. limit do
        // If i is already marked (0), skip it
        if primes[i] <> 0 then
            let mutable j = i
            // Cross out all multiples of i (starting at i*i)
            while i * j <= n do
                primes[i * j] <- 0
                j <- j + 1
    // Filter all non-zero values (prime numbers)
    primes |> Array.filter (fun p -> p >= 2) |> List.ofArray

The script `setup.fsx` displays lists completely.  Therefore, we see all prime numbers up to 10,000 below.  In F$^\sharp$, the digits of a numeric literal can be grouped with underscores.

In [ ]:
primes 10_000

## Numerical Functions

F$^\sharp$ provides a comprehensive collection of standard mathematical functions.  Most of them expect floating-point numbers as arguments.  The elementary functions include:
* **Square root calculation:** `sqrt x` calculates the square root $\sqrt{x}$.

In [ ]:
sqrt 2.0

The function `sqrt` does not accept integers: As it can be applied to arguments of several floating-point types, F$^\sharp$ does not know into which type an integer should be converted.  Hence, the next cell yields an error.

In [ ]:
sqrt 2

* **Absolute value function:** `abs x` returns the absolute value $|x|$.  It works both for integers and for floating-point numbers.

In [ ]:
abs -10

* **Exponentiation:** `x ** y` for floating-point numbers and `pown x n` for an exponent `n` of type `int`.

In [ ]:
2.0 ** 8.0

In [ ]:
pown 2 8

**Rounding Functions**

Three specific functions are available for rounding floating-point numbers to integers.  Note that their results are floating-point numbers, too.  If a value of type `int` is needed, the result has to be converted using the function `int`.
1. **`floor x`:** The *floor function* $\lfloor x \rfloor$. It always rounds down to the next smaller integer.
$$ \text{floor}(x) = \max \{ n \in \mathbb{Z} \mid n \le x \} $$

In [ ]:
floor 4.8

2. **`ceil x`:** The *ceiling function* $\lceil x \rceil$. It always rounds up to the next larger integer.
$$ \text{ceil}(x) = \min \{ n \in \mathbb{Z} \mid n \ge x \} $$

In [ ]:
ceil 4.2

3. **`round x`:** Rounding to the nearest integer.  Be careful: If `x` lies exactly in the middle between two integers, `round` rounds to the nearest *even* integer.  This is called *banker's rounding*.  Hence, `round 4.5` yields `4.0`, while `round 5.5` yields `6.0`.

In [ ]:
round 4.5

In [ ]:
round 5.5

In [ ]:
round 4.49

Commercial rounding, which always rounds up starting from $.5$, is done by the method `System.Math.Round` if its second argument is `System.MidpointRounding.AwayFromZero`.

In [ ]:
System.Math.Round(4.5, System.MidpointRounding.AwayFromZero)

**Trigonometric Functions**

F$^\sharp$ also includes the standard trigonometric functions `sin`, `cos`, and `tan` as well as their inverse functions `asin`, `acos`, and `atan`.  Note that these functions expect or return angles in radians.

In [ ]:
sin 3.14159256

**Exponential and Logarithmic Functions**

The exponential function $e^x$ is calculated via `exp x`.  Its inverse, the natural logarithm $\ln(x)$, is called `log x` in F$^\sharp$.

In [ ]:
exp 1.0

In [ ]:
log 2.718281828459045

**Constants**

Important mathematical constants are provided by the class `System.Math`:
* `System.Math.PI = `$\pi$

   This is the ratio of a circle's perimeter to its diameter.
* `System.Math.E  = `$\displaystyle\sum\limits_{n=0}^{\infty} \frac{1}{n!}$

  This is [Euler's number](https://en.wikipedia.org/wiki/E_(mathematical_constant)).

In [ ]:
System.Math.PI

In [ ]:
System.Math.E

**Functions as Values**

In F$^\sharp$, functions are values: They can be bound to variables, passed as arguments to other functions, and returned as results.  For example, the next cell binds the variable `roundDown` to the function `floor`.  The type annotation `float -> float` states that `roundDown` takes a `float` and returns a `float`.

In [ ]:
let roundDown : float -> float = floor

In [ ]:
roundDown 4.6